# Step 12: external validation on JABD (human-annotated job-ad corpus)

**Purpose.** Our PESO labels are rule-made (silver). JABD [27] has human token annotations on English job ads (EMSCAD, 2012-2014). We (a) apply the PESO-built detectors to JABD without retraining, and (b) cross-validate a TF-IDF model on JABD.

**Limits stated up front**
- JABD is a different population (US/UK English, 2012-2014; not Philippine), lemmatised and lower-cased, and its sentences were chosen by a lexicon pre-annotation, so class prevalence is far higher than in PESO. Precision is therefore **not comparable** with PESO.
- The label schemes differ (see the mapping cell). Marital status, appearance and "Female/Male" wording have no JABD class. Ethnicity, religion and disability have no rule in our pipeline.
- This is external validation of the detectors. It does not validate the PESO silver labels.
- JABD licence (Open RAIL-D): use restrictions travel with derivatives; this is research use for bias detection.

In [1]:
import json, re, platform, sys
from pathlib import Path
import numpy as np, pandas as pd, sklearn, scipy, joblib
from scipy.stats import binomtest
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import precision_recall_fscore_support, f1_score

SEED, N_SPLITS, N_BOOT = 42, 5, 1000
OUT = Path("outputs/equal_step12_jabd_validation"); OUT.mkdir(parents=True, exist_ok=True)
JABD = Path("external/jabd/jabd_annotations_v2.csv.gz")
env = dict(python=platform.python_version(), platform=platform.platform(), numpy=np.__version__, pandas=pd.__version__,
           scikit_learn=sklearn.__version__, scipy=scipy.__version__, seed=SEED, cv="GroupKFold(5) by job_id",
           bootstrap=f"{N_BOOT} resamples of job_id clusters", jabd_file=JABD.name, jabd_size_bytes=JABD.stat().st_size)
json.dump(env, open(OUT / "step12_environment.json", "w"), indent=2); print(json.dumps(env, indent=2))

{
  "python": "3.12.9",
  "platform": "Windows-11-10.0.26200-SP0",
  "numpy": "2.5.4",
  "pandas": "3.0.6",
  "scikit_learn": "1.9.1",
  "scipy": "1.18.1",
  "seed": 42,
  "cv": "GroupKFold(5) by job_id",
  "bootstrap": "1000 resamples of job_id clusters",
  "jabd_file": "jabd_annotations_v2.csv.gz",
  "jabd_size_bytes": 3772409
}


## 1. Load JABD and map labels to our scheme

Sentence-level `text_labels2` is used. Mapping (our design decision, documented here):

| Our class | JABD labels | Note |
|---|---|---|
| explicit | Explicit Gender Bias, Explicit Age Bias | priority over implicit when both are present |
| implicit | Implicit Gender Bias, Implicit Age Bias | masculine/feminine coded and young/old coded wording |
| none | no bias label at all | |
| excluded | only Explicit Ethnicity / Disability / Religion | outside our scope; neither rule nor model was designed for them |

Marital status and appearance have no JABD class, so a PESO rule firing on them counts as a false positive here. The report therefore also shows the rules with the marital component removed.

In [2]:
raw = pd.read_csv(JABD)
def labs(s): return set(json.loads(s))
L2 = raw.text_labels2.apply(labs)
def gold(s):
    if s & {"Explicit Gender Bias", "Explicit Age Bias"}: return "explicit"
    if s & {"Implicit Gender Bias", "Implicit Age Bias"}: return "implicit"
    if not s: return "none"
    return "excluded"
raw["gold"] = L2.apply(gold)
raw["text"] = raw.tokens.apply(lambda s: " ".join(json.loads(s)))
print("JABD sentences:", len(raw), "| ads:", raw.job_id.nunique())
print(raw.gold.value_counts().to_string())
d = raw[raw.gold != "excluded"].reset_index(drop=True)
print("\nin scope:", len(d), "| ads:", d.job_id.nunique())
print((d.gold.value_counts(normalize=True).round(3)).to_string())
print("\nannotation_type in scope:\n", d.annotation_type.value_counts().to_string())

JABD sentences: 14960 | ads: 8993
gold
implicit    7177
none        3385
explicit    2912
excluded    1486

in scope: 13474 | ads: 8428
gold
implicit    0.533
none        0.251
explicit    0.216

annotation_type in scope:
 annotation_type
Single Review              11986
Consolidated Review         1131
Preannotation Confirmed      225
Golden Standard              132


## 2. PESO rules applied to JABD (no retraining)

The regexes and stems are copied from Step 6/9 (the same rules that made the PESO silver labels). JABD text is lemmatised ("women" becomes "woman"), so we report the rules as written and also a lemma-adapted variant that adds the lemma forms. The explicit rule is split into three parts so each can be read separately.

In [3]:
MASC = ("active adventurous aggress ambitio analy assert athlet autonom boast challeng compet confident "
        "courag decisive decision determin dominan driven fearless force headstrong hierarch impulsive "
        "independen individual intellect lead logic objective opinion outspoken persist self-relian "
        "self-sufficien stubborn superior unreasonab").split()
FEM = ("agree affectionate child cheer collab commit communal compassion connect considerate cooperat "
       "depend emotiona empath feel flatterable gentle honest interpersonal interdependen kind kinship "
       "loyal modest nurtur pleasant polite quiet respons sensitiv submissive support sympath tender "
       "together trust understand warm enthusias inclusive yield share").split()
FP = ("confidential", "committee", "shareholder", "kinds")
def coded(text, stems):
    n = 0
    for w in re.findall(r"[a-z][a-z-]*", text.lower()):
        if w.startswith(FP): continue
        parts = [w] + (w.split("-") if "-" in w else [])
        if any(p.startswith(s) for p in parts for s in stems): n += 1
    return n
MALE_RE = re.compile(r"\b(?:male|males|men|gentlemen)\b", re.I)
FEMALE_RE = re.compile(r"\b(?:female|females|women|ladies)\b", re.I)
MALE_L = re.compile(r"\b(?:male|males|men|man|gentlemen|gentleman)\b", re.I)
FEMALE_L = re.compile(r"\b(?:female|females|women|woman|ladies|lady)\b", re.I)
INCL_RE = re.compile(r"\b(?:either\s+gender|any\s+gender|regardless\s+of\s+(?:sex|gender)|men\s+and\s+women|"
                     r"gender\s*[:\-]?\s*(?:any|open|not\s+required))\b", re.I)
MARITAL_RE = re.compile(r"\b(?:single|married|unmarried|widow(?:ed|er)?|marital|pregnan\w*)\b", re.I)
AGE_RE = re.compile(r"\b(?:\d{2}\s*(?:-|to)\s*\d{2}\s*(?:years|yrs|yo|y/o|years\s+old)|"
                    r"(?:below|under|not\s+(?:more|older)\s+than|up\s+to|maximum\s+(?:age\s+)?of)\s*\d{2}|"
                    r"age[sd]?\s*(?:range|limit|requirement)?\s*[:\-]?\s*\d{2})\b", re.I)
APPEAR_RE = re.compile(r"\b(?:good\s+looking|pleasing\s+personality|presentable|attractive|"
                       r"height|weight|physically\s+fit|heavy\s+lifting|tall|slim)\b", re.I)

def parts(t, lemma):
    m, f = (MALE_L, FEMALE_L) if lemma else (MALE_RE, FEMALE_RE)
    incl = bool(INCL_RE.search(t))
    return dict(gender=bool((m.search(t) or f.search(t)) and not incl), marital=bool(MARITAL_RE.search(t)),
                age=bool(AGE_RE.search(t)), implicit=(coded(t, MASC) + coded(t, FEM) > 0) or bool(APPEAR_RE.search(t)))
def rule_pred(t, lemma=False, marital=True):
    p = parts(t, lemma)
    if p["gender"] or p["age"] or (marital and p["marital"]): return "explicit"
    return "implicit" if p["implicit"] else "none"

d["r_asis"] = d.text.apply(rule_pred)
d["r_lemma"] = d.text.apply(lambda t: rule_pred(t, lemma=True))
d["r_lemma_nomarital"] = d.text.apply(lambda t: rule_pred(t, lemma=True, marital=False))
P = pd.DataFrame(list(d.text.apply(lambda t: parts(t, True))))
print("units where each explicit rule component fires (lemma-adapted):", P[["gender", "marital", "age"]].sum().to_dict())

units where each explicit rule component fires (lemma-adapted): {'gender': 172, 'marital': 183, 'age': 59}


## 3. PESO-trained model applied to JABD, and a JABD cross-validated model

- **Transfer:** the saved Step 9 bundle (TF-IDF + logistic regression, trained on all PESO silver labels) predicts JABD text. No retraining.
- **JABD CV:** a TF-IDF + logistic regression (`class_weight="balanced"`, same settings as Step 9) with GroupKFold by `job_id`, so no advertisement is in both train and test.

In [4]:
bundle = joblib.load("outputs/equal_step9_silver_baseline/step9_silver_model.joblib")
cls = list(bundle["classifier"].classes_)
Xp = bundle["vectorizer"].transform(d.text)
d["m_transfer"] = [cls[i] for i in bundle["classifier"].predict_proba(Xp).argmax(1)]

CL = ["none", "implicit", "explicit"]
y = d.gold.map({c: i for i, c in enumerate(CL)}).values
pred = np.zeros(len(d), int)
for tr, te in GroupKFold(n_splits=N_SPLITS).split(d, y, d.job_id.values):
    v = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    m = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED).fit(v.fit_transform(d.text.values[tr]), y[tr])
    pred[te] = m.predict(v.transform(d.text.values[te]))
d["m_jabd_cv"] = [CL[i] for i in pred]
print("done")

done


## 4. Results with job-level bootstrap intervals

Precision, recall and F1 per class, against JABD human labels. The 95% intervals resample whole advertisements (job_id) 1,000 times, because sentences from one ad are not independent.

In [5]:
jobs = d.job_id.values
uj, inv = np.unique(jobs, return_inverse=True)
def counts(col):
    out = np.zeros((len(uj), 3, 3))   # job x class x (tp, fp, fn)
    for i, c in enumerate(CL):
        g, p = (d.gold == c).values, (d[col] == c).values
        np.add.at(out[:, i, 0], inv, g & p); np.add.at(out[:, i, 1], inv, ~g & p); np.add.at(out[:, i, 2], inv, g & ~p)
    return out
def prf(c):
    tp, fp, fn = c[..., 0], c[..., 1], c[..., 2]
    p = np.divide(tp, tp + fp, out=np.full_like(tp, np.nan), where=(tp + fp) > 0)
    r = np.divide(tp, tp + fn, out=np.full_like(tp, np.nan), where=(tp + fn) > 0)
    f = np.divide(2 * p * r, p + r, out=np.zeros_like(tp), where=(p + r) > 0)
    return p, r, f
rng = np.random.default_rng(SEED)
W = rng.multinomial(len(uj), np.ones(len(uj)) / len(uj), size=N_BOOT)
systems = {"rules_as_written": "r_asis", "rules_lemma_adapted": "r_lemma", "rules_lemma_no_marital": "r_lemma_nomarital",
           "step9_model_transfer": "m_transfer", "jabd_tfidf_lr_cv": "m_jabd_cv"}
rows = []
for name, col in systems.items():
    c = counts(col); p, r, f = prf(c.sum(0)[None])
    bs = [prf(np.tensordot(w, c, axes=(0, 0))[None]) for w in W[:N_BOOT]]
    bp, br, bf = (np.array([b[k][0] for b in bs]) for k in range(3))
    for i, cl in enumerate(CL):
        if cl == "none": continue
        ci = lambda a: f"{np.nanpercentile(a[:, i], 2.5):.3f}-{np.nanpercentile(a[:, i], 97.5):.3f}"
        rows.append(dict(system=name, cls=cl, precision=round(p[0, i], 3), precision_CI=ci(bp), recall=round(r[0, i], 3), recall_CI=ci(br),
                         F1=round(f[0, i], 3), F1_CI=ci(bf)))
    rows.append(dict(system=name, cls="macro_F1(3 classes)", F1=round(f1_score(d.gold, d[col], average="macro", labels=CL), 3)))
res = pd.DataFrame(rows); res.to_csv(OUT / "step12_jabd_results.csv", index=False)
display(res.fillna(""))
print("gold counts:", d.gold.value_counts().to_dict(), "| chance reference: always-none macro-F1 =",
      round(f1_score(d.gold, ["none"] * len(d), average="macro", labels=CL), 3))

,system,cls,precision,precision_CI,recall,recall_CI,F1,F1_CI
0,rules_as_written,implicit,0.674,0.659-0.688,0.417,0.406-0.430,0.515,0.504-0.527
1,rules_as_written,explicit,0.783,0.734-0.826,0.084,0.074-0.095,0.153,0.134-0.170
2,rules_as_written,macro_F1(3 classes),,,,,0.380,
3,rules_lemma_adapted,implicit,0.678,0.663-0.693,0.417,0.405-0.429,0.516,0.505-0.528
4,rules_lemma_adapted,explicit,0.821,0.783-0.858,0.117,0.104-0.129,0.204,0.185-0.224
5,rules_lemma_adapted,macro_F1(3 classes),,,,,0.399,
6,rules_lemma_no_marital,implicit,0.675,0.661-0.690,0.418,0.407-0.431,0.516,0.505-0.528
7,rules_lemma_no_marital,explicit,0.775,0.717-0.827,0.061,0.052-0.071,0.114,0.098-0.130
8,rules_lemma_no_marital,macro_F1(3 classes),,,,,0.367,
9,step9_model_transfer,implicit,0.626,0.611-0.640,0.426,0.414-0.438,0.507,0.496-0.518


gold counts: {'implicit': 7177, 'none': 3385, 'explicit': 2912} | chance reference: always-none macro-F1 = 0.134


## 5. Is the model better than the rules for explicit bias? Paired test

For the explicit class, each unit is "correct" if the system's explicit/not-explicit decision matches the human label. McNemar's exact test compares the rules with the JABD-trained model on the same units. A significant difference shows the two differ, not that either is valid for PESO.

In [6]:
def mcnemar(a, b):
    g = (d.gold == "explicit").values
    ca, cb = ((d[a] == "explicit").values == g), ((d[b] == "explicit").values == g)
    n01, n10 = int((~ca & cb).sum()), int((ca & ~cb).sum())
    p = binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
    return dict(A=a, B=b, A_correct_B_wrong=n10, A_wrong_B_correct=n01, p_value=float(f"{p:.3g}"))
mc = pd.DataFrame([mcnemar("r_lemma_nomarital", "m_jabd_cv"), mcnemar("r_lemma_nomarital", "m_transfer"), mcnemar("m_transfer", "m_jabd_cv")])
mc.to_csv(OUT / "step12_mcnemar.csv", index=False); display(mc)

,A,B,A_correct_B_wrong,A_wrong_B_correct,p_value
0,r_lemma_nomarital,m_jabd_cv,668,1846,1.160000e-126
1,r_lemma_nomarital,m_transfer,168,53,4.020000e-15
2,m_transfer,m_jabd_cv,654,1947,5.650000e-148


## 6. Sensitivity: higher-quality labels only, and error reading

`Consolidated Review` and `Golden Standard` rows were annotated by several people or by experts, so they are the more reliable subset. Rows that the rules got wrong for the explicit class are saved so they can be read.

In [7]:
hq = d.annotation_type.isin(["Consolidated Review", "Golden Standard"])
sens = []
for name, col in systems.items():
    for lab, mask in [("all", np.ones(len(d), bool)), ("consolidated+golden", hq.values)]:
        s = d[mask]; pr, rc, f, _ = precision_recall_fscore_support(s.gold == "explicit", s[col] == "explicit", average="binary", zero_division=0)
        sens.append(dict(system=name, subset=lab, n=len(s), n_explicit=int((s.gold == "explicit").sum()), explicit_P=round(pr, 3), explicit_R=round(rc, 3), explicit_F1=round(f, 3)))
sens = pd.DataFrame(sens); sens.to_csv(OUT / "step12_sensitivity_explicit.csv", index=False); display(sens)

err = d[(d.r_lemma_nomarital == "explicit") & (d.gold != "explicit")][["job_id", "gold", "text"]]
miss = d[(d.r_lemma_nomarital != "explicit") & (d.gold == "explicit")][["job_id", "gold", "text"]]
err.to_csv(OUT / "step12_rule_false_positives_explicit.csv", index=False); miss.to_csv(OUT / "step12_rule_misses_explicit.csv", index=False)
print("rule false positives (explicit):", len(err), "| misses:", len(miss))
print("\nSample false positives:"); print(err.sample(min(8, len(err)), random_state=SEED).text.str[:140].to_string())
print("\nSample misses:"); print(miss.sample(min(8, len(miss)), random_state=SEED).text.str[:140].to_string())

,system,subset,n,n_explicit,explicit_P,explicit_R,explicit_F1
0,rules_as_written,all,13474,2912,0.783,0.084,0.153
1,rules_as_written,consolidated+golden,1263,679,0.899,0.105,0.187
2,rules_lemma_adapted,all,13474,2912,0.821,0.117,0.204
3,rules_lemma_adapted,consolidated+golden,1263,679,0.906,0.141,0.245
4,rules_lemma_no_marital,all,13474,2912,0.775,0.061,0.114
5,rules_lemma_no_marital,consolidated+golden,1263,679,0.827,0.063,0.118
6,step9_model_transfer,all,13474,2912,1.000,0.004,0.008
7,step9_model_transfer,consolidated+golden,1263,679,1.000,0.003,0.006
8,jabd_tfidf_lr_cv,all,13474,2912,0.750,0.672,0.709
9,jabd_tfidf_lr_cv,consolidated+golden,1263,679,0.866,0.918,0.891


rule false positives (explicit): 52 | misses: 2733

Sample false positives:
3620     lifting , pulling , pushing : exert up to 50 p...
9465     minimum requirements : minimum of 6 month cust...
11894    physical demandsrequires the ability to walk ,...
2443     minimum requirements : minimum of 6 month cust...
10187    ability bend , crouch , crawl ability to lift ...
1055      have create thousand of job for at-risk woman...
3195     the ideal candidate must be able to demonstrat...
12916    our designer brand include such leading name a...

Sample misses:
3681     tidewater finance company be an equal opportun...
7844     résumé or curriculum vitaecover letter ‘ o ’ l...
11211                                 senior ios developer
3417     tidewater finance company be an equal opportun...
190      we be fund by the world ’ s lead vcs , benchma...
4441     we provide love care for child & amp ; senior ...
482      we be look for someone who have live and breat...
6905     honorable disc

## 7. Reading guide
- Precision on JABD depends on its enriched prevalence, so use it for comparing systems on the same data, not for comparison with PESO.
- JABD labels were seeded by a lexicon, so recall for terms outside that lexicon is not measured, and implicit scores partly reflect lexicon overlap with our Gaucher stems.
- The rules were designed for Philippine postings, so this is a transfer test, and the lemmatised text affects them.
- No claim is made about the PESO silver labels' correctness. Human labels on PESO text remain pending.
- No transformer was trained. A RoBERTa/BERT comparison would use the same `job_id` folds and the same measures.

## 8. Why rule recall is low: definition mismatch check

JABD's explicit gender class includes any marking of sex or generic pronouns, and its explicit age class includes age-group mentions. Our explicit class is a stated restriction. This cell shows which JABD leaf classes the rules catch.

In [8]:
# Which JABD explicit leaf classes do the rules catch? (explicit-gold units only; a unit can carry several leaf classes)
import collections
leaf = pd.read_csv(JABD, usecols=["phrase_id", "text_labels4"]).set_index("phrase_id").text_labels4
d["leaf"] = d.phrase_id.map(leaf).apply(lambda s: set(json.loads(s)))
rows = []
for lc in ["Explicit Marking of Sex", "Generic He", "Generic She", "Age Related"]:
    s = d[d.leaf.apply(lambda x: lc in x) & (d.gold == "explicit")]
    rows.append(dict(jabd_leaf_class=lc, units=len(s), caught_by_rules_lemma_no_marital=int((s.r_lemma_nomarital == "explicit").sum()),
                     recall=round((s.r_lemma_nomarital == "explicit").mean(), 3)))
leafres = pd.DataFrame(rows); leafres.to_csv(OUT / "step12_rule_recall_by_leaf.csv", index=False); display(leafres)
print("\nExample JABD explicit-gold units the rules missed, by leaf class:")
for lc in ["Explicit Marking of Sex", "Generic He", "Age Related"]:
    s = d[d.leaf.apply(lambda x: lc in x) & (d.gold == "explicit") & (d.r_lemma_nomarital != "explicit")]
    print("\n", lc); print(s.text.str[:130].sample(min(4, len(s)), random_state=SEED).to_string())

,jabd_leaf_class,units,caught_by_rules_lemma_no_marital,recall
0,Explicit Marking of Sex,818,157,0.192
1,Generic He,644,6,0.009
2,Generic She,503,13,0.026
3,Age Related,1577,46,0.029



Example JABD explicit-gold units the rules missed, by leaf class:

 Explicit Marking of Sex
10691                   send us your resume and let talk .
3391     all qualified applicant will receive considera...
835      tidewater finance company be an equal opportun...
6331     new york state journeyman certification or equ...

 Generic He
1417     he or she handle guest 's check-in and checkou...
1141     he aguilar group be actively recruit for a man...
231      he or she shall assess different information t...
10619    he or she shall assess different information t...

 Age Related
5391     instrumental be a highly dynamic , fast-paced ...
491      westview financial services be an equal opport...
3150     united cerebral palsy of oregon & amp ; sw was...
11218    nuneaton dgv apprenticeship available under na...
